<a href="https://colab.research.google.com/github/Mizharrrrrhidi1818/EnsembleMethod-XGBoost-SalesForcasting/blob/main/Notebooks/Ensemble_Method_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Introduction**

In modern business analytics, accurately forecasting sales performance is critical for inventory management, budget allocation, and strategic planning. Traditional statistical models often struggle with complex, non-linear relationships, missing values, and high-dimensional categorical features. Machine learning, particularly ensemble tree-based methods, has emerged as the industry standard for structured tabular data.

Among these, XGBoost (eXtreme Gradient Boosting) stands out as a highly optimized, scalable, and mathematically rigorous algorithm that consistently wins Kaggle competitions and powers production systems worldwide. This report presents a complete end-to-end implementation of XGBoost applied to a sales forecasting dataset. We explore both regression (predicting continuous sales values) and classification (predicting whether an order belongs to a high- or low-sales category), comparing a from-scratch mathematical implementation with the industry-standard library. The goal is to provide deeply explainable walkthrough of how XGBoost works, why it performs so well, and how to deploy it responsibly in real-world scenarios.

# **2. Methodology**

Our workflow follows a structured, reproducible machine learning pipeline:
1. Data Acquisition: Download the transactional sales dataset from Kaggle.
2. Preprocessing & Cleaning: Handle missing values, remove duplicates, convert data types, and drop non-predictive identifiers.
3. Feature Engineering: Aggregate transactions by order, create temporal features (weekday, month), discretize sales into interpretable bins, and one-hot encode categorical variables.
4. Target Formulation:
    - Regression: Use raw aggregated sales as a continuous target.
    - Classification: Convert sales bins into a binary label (0 = Low Sales, 1 = High Sales).
5. Model Development:
    - Build a custom XGBoost implementation from scratch using NumPy to expose the underlying mathematics.
    - Train the official xgboost.XGBClassifier/XGBRegressor for benchmarking.
6. Evaluation: Compare predictions using domain-appropriate metrics (RMSE for regression, Accuracy/Precision/Recall/F1 for classification).
7. Analysis & Interpretation: Contrast mathematical theory with empirical results, discuss trade-offs, and outline production considerations.

This methodology balances educational transparency (custom code) with production readiness (library implementation), ensuring both conceptual understanding and practical applicability.

# **3. Objective**

The primary objectives of this project are:
- Predictive Accuracy: Develop robust models that can reliably forecast sales magnitude and classify order performance.
- Algorithmic Transparency: Demystify XGBoost by implementing its core optimization steps (gradient descent on trees, Taylor expansion, regularization) from first principles.
- Comparative Analysis: Evaluate the performance gap between a pedagogical implementation and a highly optimized library version.
- Business Relevance: Translate model outputs into actionable insights (e.g., identifying high-value orders, understanding seasonal/segment-driven sales patterns).
- Reproducibility: Provide a clean, well-documented pipeline that students and practitioners can adapt, extend, and deploy in similar retail or e-commerce contexts.

# **4. Data Exploration and Preprocessing**

## **4.1. Dataset Overview**

In [ ]:
import pandas as pd, numpy as np
from pandas import DataFrame, Series
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [1]:
# 1. Load dataset anda data preprocessing
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohitsahoo/sales-forecasting")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sales-forecasting' dataset.
Path to dataset files: /kaggle/input/sales-forecasting


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# This know our dataset after we have finished download

/kaggle/input/sales-forecasting/train.csv


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
import tabulate
warnings.filterwarnings("ignore")

# Use the 'path' variable from the previous download cell to construct the correct file path
df = pd.read_csv(os.path.join(path, "train.csv"))

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

The dataset contains historical transactional records for a retail
- business. Key attributes include:
- Identifiers: `Row ID`, `Order ID`
- Financial: `Sales` (continuous target)
- Temporal: `Order Date`, `Ship Date`
- Geographic: `City, State, Postal Code, Region`
- Customer & Product: `Segment, Category, Sub-Category, Ship Mode`

Each row represents a line item within an order. Since multiple items can belong to the same Order ID, the raw data is not at the order level. Predicting at the item level would inflate sample size artificially and ignore order-level aggregation effects. Therefore, we aggregate by Order ID to align features and targets at the correct business unit.

## **4.2. Data Cleaning**

Data quality directly dictates model reliability. We performed the following cleaning steps:

In [ ]:
# Check missing values
print(df.isnull().sum())

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64


In [ ]:
# Impute Missing Values(Postal code) with a known Vermont ZIP (e.g., Burlington, VT = 05401)
zip = df["Postal Code"].isnull() & \
       (df['City'] == 'Burlington') & \
       (df['State'] == 'Vermont')

df.loc[zip, 'Postal Code'] = '05401'

#Verify
print("Missing after imputation:", zip.isnull().sum())
print(df.isnull().sum())

Missing after imputation: 0
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
dtype: int64


Missing Value Imputation: `Postal Code` contained nulls for Burlington, Vermont. Using geographic logic, we imputed `05401` (Burlington's known ZIP). This preserves sample size without introducing artificial noise.

In [ ]:
# Convert Order Date and Ship Date to datetime with dayfirst=True
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst = True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst = True)

# Drop 'Row ID' – not meaningful
df.drop(columns=["Row ID"], inplace=True)

- Type Conversion: Order Date and Ship Date were parsed into proper datetime objects with dayfirst=True to avoid US/EU format ambiguity.

- Column Dropping: Row ID was removed as it is a surrogate key with zero predictive value.

In [ ]:
# Detect duplicate
df.duplicated().any()

np.True_

In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
duplicates = df[df.duplicated()]

duplicates

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
3406,US-2015-150119,2015-04-23,2015-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372


In [ ]:
# Removing duplicate
df = df.drop_duplicates()

df.duplicated().any()

np.False_

Duplicate Removal: Exact row duplicates were identified and dropped to prevent data leakage and inflated performance metrics.

## **4.3. Feature Engineering**

Feature engineering involves creating new features or modifying existing ones to improve the performance of machine learning models.

In [ ]:
# Aggregate by Order ID
order_df = df.groupby("Order ID").agg({
    "Sales": "sum",
    "Order Date": "first",
    "Ship Mode": "first",
    "Segment": "first",
    "Region": "first",
    "Category": lambda x: ', '.join(set(x)),
    "Sub-Category": lambda x: ', '.join(set(x))
}).reset_index()

# Temporal features
order_df["Order_Weekday"] = order_df["Order Date"].dt.day_name()
order_df["Order_Month"] = order_df["Order Date"].dt.month_name().str[:3]

# Discretize Sales into bins
order_df["Sales_bin"] = pd.cut(
    order_df["Sales"],
    bins=[0, 100, 300, 600, 10000],
    labels=["[0,100)", "[100,300)", "[300,600)", "[600+,10k]"],
    include_lowest=True
)

# Create Binary Classification Target: 0 = Low Sales, 1 = High Sales
order_df['Sales_binary'] = order_df['Sales_bin'].apply(
    lambda x: 1 if x in ["[300,600)", "[600+,10k]"] else 0
)

# One-Hot Encoding
features = ["Order_Weekday", "Order_Month", "Ship Mode", "Segment", "Region", "Category"]
X = pd.get_dummies(order_df[features], drop_first=True)
y = order_df['Sales_binary']

# Remove rows with NaNs
mask = ~y.isnull() & ~X.isnull().any(axis=1)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True).astype(int)  # Ensure integer labels

print("Final shape:", X.shape)
print("Target Distribution:\n", y.value_counts())

Final shape: (4916, 33)
Target Distribution:
 Sales_binary
0    3143
1    1773
Name: count, dtype: int64


Raw features rarely translate directly into model-ready signals. We engineered the following:
- Order-Level Aggregation: Grouped by Order ID and summed Sales. For categorical features (Segment, Region, Ship Mode), we took the first occurrence (consistent per order). For Category and Sub-Category, we joined unique values to preserve multi-category orders.
- Temporal Features: Extracted Order_Weekday (day of week) and Order_Month (abbreviated month). These capture seasonality, weekend vs. weekday purchasing behavior, and promotional cycles.
- Target Discretization: Sales were binned into [0,100), [100,300), [300,600), [600+,10k]. For classification, we mapped the two upper bins to 1 (High Sales) and lower bins to 0 (Low Sales).
- One-Hot Encoding: Categorical features were transformed into binary indicator columns using pd.get_dummies(drop_first=True) to avoid multicollinearity while preserving all information

We will create encode the categorical features numerically that is new binary target from `Sales_bin` and one-hot encode the other features.

## **4.4. Dataset Training and Testing**

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train_np = X_train.to_numpy(dtype=float)
X_test_np = X_test.to_numpy(dtype=float)

- Split Strategy: 70% training, 30% testing (train_test_split with random_state=42 for reproducibility).
- NaN Handling: Post-engineering rows with missing features or targets were removed to ensure clean matrix operations.
- Data Types: Converted DataFrames to NumPy arrays of float64 for numerical stability in gradient computations.
- Class Balance: The binary target distribution was inspected. If heavily imbalanced, techniques like scale_pos_weight or stratified splitting would be applied (not required here, but noted for production readiness).

# **5. XGBoost Regression**

## **5.1. XGboost Regression Formula**

XGBoost is an additive ensemble of decision trees. Unlike Random Forests (which average independent trees), XGBoost builds trees sequentially, where each new tree corrects the residuals (errors) of the previous ensemble.

The core objective function for regression is:

**Objective Function:**
$$ \mathcal{L}(\theta) = \sum_{i=1}^{n} l(y_i, \hat{y}_i^{(t)}) + \sum_{k=1}^{t} \Omega(f_k) $$

Where is
- $ \mathcal{L}(\theta) $ is the loss function (Mean Squared Error: $ 1/2(y_i-y ̂_i )^2$ )
- $ y ̂_i^((t) )=y ̂_i^(tⓜ-1) +f_t (x_i) $ is the prediction after trees
- $ Ω(f_k) $ is the regularization term penalizing tree complexity (number of leaves + squared leaf weights)

Because optimizing this directly is computationally intractable, XGBoost uses a second-order Taylor expansion around the current prediction:

**Second-Order Taylor Expansion:**
$$ \mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t) $$

Where:
- $ g_i= \frac {∂l} {∂ \hat{y}_i^{(t-1)}}= \hat{y}_i^{(t-1)} - y_i $ (first derivative / gradient)
- $ h_i= \frac {∂^2 l} {∂\hat{y}_i^{(t-1)2}} =1 $(second derivative / Hessian for MSE)


**Optimal Leaf Weight:**

Each leaf `j` gets an optimal weight:

$$ w_j^* = -\frac{G_j}{H_j + \lambda} $$


**Split Gain Formula:**
$$ \text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda} \right] - \gamma $$

Where $ γ $ penalizes adding new leaves. If Gain > 0, the split is accepted. This mathematically guarantees that every split improves the objective function.

In [ ]:
# Xboost Regression
class XGBoostTree:
    def __init__(self, x, g, h, params, depth=0):
        self.x = x
        self.g = g
        self.h = h
        self.params = params
        self.depth = depth
        self.is_leaf = False
        self.val = None
        self.left = None
        self.right = None

        # Initialize these attributes here
        self.split_col = None
        self.split_val = None
        # ----------------------------------------------

        # Calculate leaf weight
        self.val = -np.sum(g) / (np.sum(h) + params['lambda'])

        if depth < params['max_depth']:
            self._find_split()

    def _find_split(self):
        best_gain = 0
        self.split_col = None
        self.split_val = None

        # Current Similarity Score
        G, H = np.sum(self.g), np.sum(self.h)
        current_score = (G ** 2) / (H + self.params['lambda'])

        for col in range(self.x.shape[1]):
            x_col = self.x[:, col]
            for val in np.unique(x_col):
                left_mask = x_col <= val
                right_mask = ~left_mask

                if not any(left_mask) or not any(right_mask):
                    continue

                G_l, H_l = np.sum(self.g[left_mask]), np.sum(self.h[left_mask])
                G_r, H_r = np.sum(self.g[right_mask]), np.sum(self.h[right_mask])

                gain = 0.5 * ((G_l ** 2 / (H_l + self.params['lambda'])) +
                        (G_r ** 2 / (H_r + self.params['lambda'])) -
                        current_score) - self.params['gamma']

                if gain > best_gain:
                    best_gain, self.split_col, self.split_val = gain, col, val

        if self.split_col is not None:
            left_mask = self.x[:, self.split_col] <= self.split_val
            self.left = XGBoostTree(self.x[left_mask], self.g[left_mask], self.h[left_mask], self.params,
                                    self.depth + 1)
            self.right = XGBoostTree(self.x[~left_mask], self.g[~left_mask], self.h[~left_mask], self.params,
                                     self.depth + 1)
        else:
            self.is_leaf = True

    def predict(self, x):
        if self.is_leaf or self.split_col is None:
            return self.val
        if x[self.split_col] <= self.split_val:
            return self.left.predict(x)
        return self.right.predict(x)


class CustomXGBoostRegressor:
    def __init__(self, n_estimators=10, max_depth=3, learning_rate=0.1, reg_lambda=1.0, gamma=0.0):
        self.params = {'max_depth': max_depth, 'lambda': reg_lambda, 'gamma': gamma}
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.trees = []

    def _get_gradients(self, y, y_pred):
        return y_pred - y, np.ones_like(y)

    def fit(self, X, y):
        y_pred = np.zeros_like(y, dtype=float)
        for _ in range(self.n_estimators):
            g, h = self._get_gradients(y, y_pred)
            tree = XGBoostTree(X, g, h, self.params)
            update = np.array([tree.predict(row) for row in X])
            y_pred += self.lr * update
            self.trees.append(tree)

    def predict(self, X):
        y_pred = np.zeros(X.shape[0])
        for tree in self.trees:
            y_pred += self.lr * np.array([tree.predict(row) for row in X])
        return y_pred

## **5.2. Evaluation Metrics (RMSE)**

For regression, we use Root Mean Squared Error (RMSE):

$$ \text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} $$

- Why RMSE? It penalizes larger errors quadratically, making the model sensitive to outliers (important in sales where high-value orders significantly impact revenue).
- Interpretation: RMSE is in the same units as the target. A lower RMSE indicates tighter alignment between predicted and actual sales.
- Connection to XGBoost: Minimizing MSE directly aligns with XGBoost's gradient computation $ (g_i = \hat{y}_i - y_i) $ , ensuring mathematical consistency between training objective and evaluation metric.



In [ ]:
# Convert DataFrames to NumPy arrays and explicitly ensure float dtype for the custom model
X_train_np = X_train.to_numpy(dtype=float)
X_test_np = X_test.to_numpy(dtype=float)
y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

# Custom Model
custom_model = CustomXGBoostRegressor(n_estimators=5, max_depth=3, learning_rate=0.1)
custom_model.fit(X_train_np, y_train_np)
custom_preds = custom_model.predict(X_test_np)
custom_rmse = np.sqrt(mean_squared_error(y_test_np, custom_preds))

#Scikit-learn (XGBoost Library)
sklearn_xgb = XGBRegressor(n_estimators=5, max_depth=3, learning_rate=0.1,
                               reg_lambda=1, gamma=0, base_score=0)
sklearn_xgb.fit(X_train, y_train) # Keep original DataFrames for sklearn model
sklearn_preds = sklearn_xgb.predict(X_test)
sklearn_rmse = np.sqrt(mean_squared_error(y_test, sklearn_preds))

print(f"Custom Implementation RMSE: {custom_rmse:.4f}")
print(f"Sklearn RMSE:     {sklearn_rmse:.4f}")

Custom Implementation RMSE: 0.4872
Sklearn RMSE:     0.4872


# **6. XGBoost Classification**

## **6.1. XGBoost Classification Formula**

Classification shifts the goal from predicting a continuous value to predicting a probability that an instance belongs to class 1. The loss function changes from MSE to Binary Cross-Entropy (Log Loss):

**Binary Cross-Entropy (Log Loss):**
$$ l(y_i, \hat{y}_i) = - \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right] $$

Where $ p_i= σ(y ̂_i) = \frac {1} {1 + e^\hat{-y}_i} $ is the sigmoid function mapping raw logits to probabilities.

**Gradients & Hessians (Classification):**

Using the chain rule, the first and second derivatives become:
$$ g_i = p_i - y_i, \quad h_i = p_i (1 - p_i) $$

Notice the elegant similarity to regression: the tree structure, leaf weight formula, and split gain remain identical. Only the gradient/Hessian definitions change. This is XGBoost's core strength: a unified optimization framework that adapts to different tasks by swapping the loss function's derivatives.

Prediction Pipeline:
1. Initialize predictions at 0 (log-odds of 50% probability).
2. Iteratively fit trees to residuals defined by $ g_i, h_i $.
3. Sum tree outputs: $ \hat{y}_i $
4. Apply sigmoid: $ p_i = \sigma(\hat{y}_i) $.
5. Threshold at 0.5: "class" =  1 if $ p_i ≥ 0.5 $, else 0.

This probabilistic approach allows business users to adjust decision thresholds based on risk tolerance (e.g., lowering threshold to 0.3 to capture more potential high-sales orders at the cost of more false positives).



In [ ]:
# Custom XGBoost CLASSIFIER (From Scratch)
def sigmoid(x):
    # Numerically stable sigmoid
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

class XGBoostTreeClf:
    def __init__(self, x, g, h, params, depth=0):
        self.x, self.g, self.h = x, g, h
        self.params, self.depth = params, depth
        self.is_leaf = False
        self.split_col, self.split_val = None, None
        self.left, self.right = None, None
        self.val = None

        if depth < params['max_depth']:
            self._find_split()
        else:
            self.is_leaf = True

        if self.is_leaf:
            # Newton-Raphson step for leaf weight
            self.val = -np.sum(g) / (np.sum(h) + params['lambda'])

    def _find_split(self):
        best_gain = 0
        G, H = np.sum(self.g), np.sum(self.h)
        current_score = (G ** 2) / (H + self.params['lambda'])

        for col in range(self.x.shape[1]):
            x_col = self.x[:, col]
            for val in np.unique(x_col):
                left_mask = x_col <= val
                right_mask = ~left_mask
                if not any(left_mask) or not any(right_mask):
                    continue

                G_l, H_l = np.sum(self.g[left_mask]), np.sum(self.h[left_mask])
                G_r, H_r = np.sum(self.g[right_mask]), np.sum(self.h[right_mask])

                gain = 0.5 * ((G_l**2 / (H_l + self.params['lambda'])) +
                              (G_r**2 / (H_r + self.params['lambda'])) - current_score) - self.params['gamma']

                if gain > best_gain:
                    best_gain, self.split_col, self.split_val = gain, col, val

        if self.split_col is not None:
            left_mask = self.x[:, self.split_col] <= self.split_val
            self.left = XGBoostTreeClf(self.x[left_mask], self.g[left_mask], self.h[left_mask], self.params, self.depth+1)
            self.right = XGBoostTreeClf(self.x[~left_mask], self.g[~left_mask], self.h[~left_mask], self.params, self.depth+1)
        else:
            self.is_leaf = True
            self.val = -np.sum(self.g) / (np.sum(self.h) + self.params['lambda'])

    def predict(self, x):
        if self.is_leaf or self.split_col is None:
            return self.val
        return self.left.predict(x) if x[self.split_col] <= self.split_val else self.right.predict(x)


class CustomXGBoostClassifier:
    def __init__(self, n_estimators=10, max_depth=3, learning_rate=0.1, reg_lambda=1.0, gamma=0.0):
        self.params = {'max_depth': max_depth, 'lambda': reg_lambda, 'gamma': gamma}
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.trees = []

    def _get_gradients(self, y, y_pred_raw):
        # STEP 2 FIX: Logistic Loss Gradients & Hessians
        p = sigmoid(y_pred_raw)
        g = p - y                  # First derivative of LogLoss
        h = p * (1 - p) + 1e-7     # Second derivative (Hessian) + epsilon for stability
        return g, h

    def fit(self, X, y):
        y_pred_raw = np.zeros(len(y), dtype=float)  # Raw logits (sum of tree outputs)
        for _ in range(self.n_estimators):
            g, h = self._get_gradients(y, y_pred_raw)
            tree = XGBoostTreeClf(X, g, h, self.params)
            update = np.array([tree.predict(row) for row in X])
            y_pred_raw += self.lr * update
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        y_pred_raw = np.zeros(X.shape[0])
        for tree in self.trees:
            y_pred_raw += self.lr * np.array([tree.predict(row) for row in X])
        return sigmoid(y_pred_raw)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

## **6.2. Evaluation Metrics (Accuracy)**

Accuracy measures the proportion of correctly classified instances:

$$ Accuracy = \frac {TP+TN} {TP+TN+FP+FN} $$

- Why Use It? It provides an intuitive, high-level view of model performance. In balanced or near-balanced sales datasets, accuracy is a reliable indicator.
- Limitations & Complementary Metrics: Accuracy can be misleading if classes are imbalanced. Therefore, we also report Precision (reliability of positive predictions), Recall (ability to find all positives), and F1-Score (harmonic mean of precision & recall). These form a complete diagnostic suite for classification tasks.




In [ ]:
# A. Custom Implementation
custom_clf = CustomXGBoostClassifier(n_estimators=10, max_depth=3, learning_rate=0.1)
custom_clf.fit(X_train_np, y_train.to_numpy())
custom_preds = custom_clf.predict(X_test_np)

# B. XGBoost Library Classifier
lib_clf = XGBClassifier(n_estimators=10, max_depth=3, learning_rate=0.1, eval_metric='logloss')
lib_clf.fit(X_train, y_train)
lib_preds = lib_clf.predict(X_test)

# C. Classification Metrics
print("\nCustom XGBoost Classifier:")
print(classification_report(y_test, custom_preds))

print("XGBoost Library Classifier:")
print(classification_report(y_test, lib_preds))

# Compare Accuracy
acc_custom = accuracy_score(y_test, custom_preds)
acc_lib = accuracy_score(y_test, lib_preds)
print(f"\n Accuracy Comparison:")
print(f"Custom Implementation : {acc_custom:.4f}")
print(f"XGBoost Library       : {acc_lib:.4f}")


Custom XGBoost Classifier:
              precision    recall  f1-score   support

           0       0.84      0.69      0.76       953
           1       0.57      0.76      0.65       522

    accuracy                           0.71      1475
   macro avg       0.71      0.72      0.70      1475
weighted avg       0.75      0.71      0.72      1475

XGBoost Library Classifier:
              precision    recall  f1-score   support

           0       0.69      0.97      0.81       953
           1       0.82      0.21      0.33       522

    accuracy                           0.70      1475
   macro avg       0.75      0.59      0.57      1475
weighted avg       0.74      0.70      0.64      1475


 Accuracy Comparison:
Custom Implementation : 0.7139
XGBoost Library       : 0.7031


Interpretation: An accuracy of 0.71 means 71% of orders were correctly classified as High or Low sales. Combined with precision/recall, stakeholders can quantify the trade-off between missed opportunities (false negatives) and wasted resources (false positives).

# **7. Conclusion**

This project demonstrates a rigorous, end-to-end application of XGBoost for sales prediction, bridging theoretical mathematics with practical implementation.

Key Takeaways:
1. Mathematical Elegance: XGBoost's power lies in its unified second-order optimization. Whether predicting continuous sales or classifying order performance, the algorithm relies on the same tree-growing mechanics, only swapping gradient/Hessian definitions based on the loss function.
2. Custom vs. Library: The from-scratch implementation successfully replicates XGBoost's core logic, proving invaluable for educational purposes. However, the official library outperforms it in speed, memory efficiency, and advanced features (histogram binning, parallelization, early stopping). In production, the library is always preferred.
3. Business Impact: By combining temporal features, customer segmentation, and product categories, XGBoost captures complex, non-linear sales drivers. The classification output enables proactive resource allocation, while regression outputs support revenue forecasting.
4. Best Practices: Proper data cleaning, order-level aggregation, and careful target formulation are just as critical as model selection. Evaluation metrics must align with business objectives, not just mathematical convenience.

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 0 — SETUP: Dataset, Preprocessing, Models, Sample
# ═══════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# Helper formatting
SEP = "="*70
def header(step_name): print(f"\n{SEP}\n{step_name}\n{SEP}")
def sub(title): print(f"\n--- {title} ---")

# 0.1 Load Dataset
print("Downloading Amazon Sales Dataset...")
path = kagglehub.dataset_download("karkavelrajaj/amazon-sales-dataset")
df = pd.read_csv(os.path.join(path, "amazon.csv"))
print(f"Dataset loaded. Shape: {df.shape}")

# 0.2 Data Cleaning & Feature Engineering
# Drop ID-like columns and target candidates to auto-detect
drop_cols = [c for c in df.columns if 'id' in c.lower() or 'url' in c.lower()]
df_clean = df.drop(columns=drop_cols, errors='ignore')

# Identify target: prefer 'category', 'purchased', or first categorical column
target_candidates = ['category', 'purchased', 'best_seller', 'stock_status']
target_col = next((c for c in target_candidates if c in df_clean.columns),
                  df_clean.select_dtypes(include=['object', 'category']).columns[0])

# Separate features & target
y = df_clean[target_col].copy()
X = df_clean.drop(columns=[target_col])

# Impute missing values
num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns
X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

# Encode categoricals
le = LabelEncoder()
y = le.fit_transform(y)
CLASSES = le.classes_
C = len(CLASSES)

X = pd.get_dummies(X, columns=cat_cols, drop_first=False).astype(float)

# Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 0.3 Train Models (all wrapped for predict_proba compatibility)
print("\nTraining classifiers...")
raw_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear Regression (Clf)": CalibratedClassifierCV(RidgeClassifier(), cv=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

MODELS = {}
ERROR = {}
for name, clf in raw_models.items():
    clf.fit(X_train_scaled, y_train)
    MODELS[name] = clf
    acc = accuracy_score(y_train, clf.predict(X_train_scaled))
    ERROR[name] = 1.0 - acc  # Training error rate for weighting
    print(f"  {name:<25} | Train Acc: {acc:.4f}")

# 0.4 Select One Ambiguous Test Object
# Ambiguity = highest entropy of averaged predictions
avg_proba = np.mean([clf.predict_proba(X_test_scaled) for clf in MODELS.values()], axis=0)
entropy = -np.sum(avg_proba * np.log(avg_proba + 1e-12), axis=1)
ambiguous_idx = np.argmax(entropy)
x_sample = X_test_scaled[ambiguous_idx:ambiguous_idx+1]  # Shape (1, n_features)
TRUE_LABEL = CLASSES[y_test[ambiguous_idx]]

print(f"\nSelected ambiguous test sample index: {ambiguous_idx}")
print(f"True class: {TRUE_LABEL}")
print(f"Features shape: {x_sample.shape}, Classes: {C}")

# ═══════════════════════════════════════════════════════════
# STEP 1 — Collect the three prediction levels
# ═══════════════════════════════════════════════════════════
header("STEP 1 — COLLECT THREE PREDICTION LEVELS")

# Abstract: hard label per classifier
ABSTRACT = {name: CLASSES[clf.predict(x_sample)[0]]
            for name, clf in MODELS.items()}

# Measurement: probability vector per classifier
PROBA = {name: clf.predict_proba(x_sample)[0]
         for name, clf in MODELS.items()}

# Rank: 1-based rank per class derived from proba
RANK = {}
for name, p in PROBA.items():
    order = np.argsort(-p)                  # indices best→worst
    r = np.empty(C, dtype=int)
    for pos, ci in enumerate(order):
        r[ci] = pos + 1
    RANK[name] = r                          # r[class_index] = rank

sub("Abstract level — one hard label per classifier")
for name, label in ABSTRACT.items():
    print(f"  {name:<25} → {{{label}}}")

sub("Measurement level — probability vector per classifier")
for name, p in PROBA.items():
    vals = tuple(round(float(v), 4) for v in p)
    print(f"  {name:<25} → {vals}")

sub("Rank level — rank per class (1 = most likely)")
header_str = f"  {'Model':<25} " + " ".join(f"{c:<12}" for c in CLASSES)
print(header_str)
for name, r in RANK.items():
    row_vals = " ".join(f"{r[i]:<12}" for i in range(C))
    print(f"  {name:<25} {row_vals}")

# ═══════════════════════════════════════════════════════════
# STEP 2 — ABSTRACT LEVEL fusion
# ═══════════════════════════════════════════════════════════
header("STEP 2 — ABSTRACT LEVEL FUSION")

# 2a. Majority Vote
sub("2a. Majority Vote")
print("  Formula: d̂(x) = argmax_v  Σᵢ I(dᵢ(x) = v)\n")
vote_counts = {cls: 0 for cls in CLASSES}
for label in ABSTRACT.values():
    vote_counts[label] += 1

print("  Votes cast by each classifier:")
for name, label in ABSTRACT.items():
    print(f"    {name:<25} → votes for '{label}'")
print(f"\n  Vote tally:")
for cls, cnt in vote_counts.items():
    print(f"    {cls:<12} : {cnt} vote(s)")

mv_winner = max(vote_counts, key=vote_counts.get)
print(f"\n  Decision (Majority Vote)  →  {{{mv_winner}}}")

# 2b. Weighted Majority Vote
sub("2b. Weighted Majority Vote")
print("  Formula: d̂(x) = argmax_v  Σᵢ wᵢ·I(dᵢ(x) = v)")
print("  Weights: wᵢ ∝ log((1 − eᵢ) / eᵢ)\n")

raw_w = {}
for name, e in ERROR.items():
    e = min(max(e, 1e-4), 1 - 1e-4)
    raw_w[name] = np.log((1 - e) / e)

total_w = sum(raw_w.values())
W = {name: raw_w[name] / total_w for name in MODELS}

print("  Weights:")
for name, w in W.items():
    print(f"    {name:<25} err={ERROR[name]:.4f}  w={w:.4f}")

wmv_scores = {cls: 0.0 for cls in CLASSES}
print("\n  Weighted votes:")
for name, label in ABSTRACT.items():
    wmv_scores[label] += W[name]
    print(f"    {name:<25} votes '{label}' with weight {W[name]:.4f}")

print(f"\n  Weighted score per class:")
for cls, sc in wmv_scores.items():
    print(f"    {cls:<12} : {sc:.4f}")

wmv_winner = max(wmv_scores, key=wmv_scores.get)
print(f"\n  Decision (Weighted Majority Vote)  →  {{{wmv_winner}}}")

# ═══════════════════════════════════════════════════════════
# STEP 3 — RANK LEVEL fusion
# ═══════════════════════════════════════════════════════════
header("STEP 3 — RANK LEVEL FUSION")

# 3a. Borda Count
sub("3a. Borda Count")
print("  Formula: Borda(vᵢ) = Σⱼ (c − rankⱼ(vᵢ))\n")
borda = np.zeros(C)
print("  Borda votes per classifier:")
for name, r in RANK.items():
    votes = C - r
    borda += votes
    vals  = {CLASSES[i]: int(votes[i]) for i in range(C)}
    print(f"    {name:<25} → {vals}")

borda_dict = {CLASSES[i]: int(borda[i]) for i in range(C)}
print(f"\n  Total Borda scores  →  {borda_dict}")
bc_winner  = CLASSES[int(np.argmax(borda))]
print(f"  Decision (Borda Count)  →  {{{bc_winner}}}")

# 3b. Highest Rank
sub("3b. Highest Rank Method")
print("  Formula: minRank(vᵢ) = min over all classifiers of rankⱼ(vᵢ)\n")
rank_mat = np.stack(list(RANK.values()), axis=0)
min_ranks = rank_mat.min(axis=0)
min_rank_dict = {CLASSES[i]: int(min_ranks[i]) for i in range(C)}
print(f"  Min rank per class  →  {min_rank_dict}")
hr_winner = CLASSES[int(np.argmin(min_ranks))]
print(f"  Decision (Highest Rank)  →  {{{hr_winner}}}")

# 3c. Intersection Method
sub("3c. Intersection Method")
print("  Training: threshold = worst rank ever given to correct class\n")
thresholds = {}
for name, clf in MODELS.items():
    proba_train = clf.predict_proba(X_train_scaled)
    worst = 0
    for i, true_ci in enumerate(y_train):
        p = proba_train[i]
        order  = np.argsort(-p)
        rank_true = int(np.where(order == true_ci)[0][0]) + 1
        if rank_true > worst: worst = rank_true
    thresholds[name] = worst

print("  Thresholds (worst training rank per classifier):")
for name, thr in thresholds.items():
    print(f"    {name:<25} threshold = {thr}")

accepted_sets = {}
print("\n  Accepted class sets for test sample:")
for name, r in RANK.items():
    thr = thresholds[name]
    accepted = {CLASSES[i] for i in range(C) if r[i] <= thr}
    accepted_sets[name] = accepted
    print(f"    {name:<25} (thr={thr}) → {accepted}")

intersection = set(CLASSES)
for s in accepted_sets.values():
    intersection &= s
if not intersection: intersection = set(CLASSES)

print(f"\n  Intersection  →  {intersection}")
inter_winner = ', '.join(sorted(intersection))
print(f"  Decision (Intersection)  →  {{{inter_winner}}}")

# 3d. Union Method
sub("3d. Union Method")
print("  Training: max-min threshold procedure\n")
union_thresholds = {}
for name, clf in MODELS.items():
    proba_train = clf.predict_proba(X_train_scaled)
    correct_ranks = []
    for i, true_ci in enumerate(y_train):
        order     = np.argsort(-proba_train[i])
        rank_true = int(np.where(order == true_ci)[0][0]) + 1
        correct_ranks.append(rank_true)
    min_rank = min(correct_ranks)
    filtered = [r if r == min_rank else 0 for r in correct_ranks]
    union_thresholds[name] = max(filtered) if any(f > 0 for f in filtered) else 1

print("  Thresholds (max-min procedure):")
for name, thr in union_thresholds.items():
    print(f"    {name:<25} threshold = {thr}")

union_sets = {}
print("\n  Accepted class sets for test sample:")
for name, r in RANK.items():
    thr = union_thresholds[name]
    accepted = {CLASSES[i] for i in range(C) if r[i] <= thr}
    union_sets[name] = accepted
    print(f"    {name:<25} (thr={thr}) → {accepted}")

union_result = set()
for s in union_sets.values():
    union_result |= s

print(f"\n  Union  →  {union_result}")
print(f"  Decision (Union)  →  {{{', '.join(sorted(union_result))}}}")

# ═══════════════════════════════════════════════════════════
# STEP 4 — MEASUREMENT LEVEL fusion
# ═══════════════════════════════════════════════════════════
header("STEP 4 — MEASUREMENT LEVEL FUSION")

L = len(MODELS)
DP = np.stack(list(PROBA.values()), axis=0)
names_list = list(MODELS.keys())

print(f"\n  Decision Profile DP(x)  —  shape ({L} classifiers × {C} classes)")
for i, name in enumerate(names_list):
    row = [f"{DP[i,j]:.4f}" for j in range(C)]
    print(f"  {name:<25}  {' | '.join(row)}")

def show_support(W_j, rule_name):
    w_dict = {CLASSES[j]: round(float(W_j[j]), 6) for j in range(C)}
    best   = CLASSES[int(np.argmax(W_j))]
    print(f"\n  Support W_j  →  {w_dict}")
    print(f"  Decision ({rule_name})  →  {{{best}}}")
    return best

# 4a. Max Rule
sub("4a. Max Rule")
print("  Formula: W_j = max over classifiers of p_{i,j}\n")
for j, cls in enumerate(CLASSES):
    vals = tuple(round(float(DP[i, j]), 4) for i in range(L))
    print(f"  col '{cls}': {vals}  →  max = {max(vals):.4f}")
show_support(DP.max(axis=0), "Max Rule")

# 4b. Min Rule
sub("4b. Min Rule")
print("  Formula: W_j = min over classifiers of p_{i,j}\n")
for j, cls in enumerate(CLASSES):
    vals = tuple(round(float(DP[i, j]), 4) for i in range(L))
    print(f"  col '{cls}': {vals}  →  min = {min(vals):.4f}")
show_support(DP.min(axis=0), "Min Rule")

# 4c. Median Rule
sub("4c. Median Rule")
print("  Formula: W_j = median over classifiers of p_{i,j}\n")
show_support(np.median(DP, axis=0), "Median Rule")

# 4d. Sum Rule
sub("4d. Sum Rule")
print("  Formula: W_j = Σᵢ p_{i,j}\n")
show_support(DP.sum(axis=0), "Sum Rule")

# 4e. Product Rule
sub("4e. Product Rule")
print("  Formula: W_j = Πᵢ p_{i,j}  (zeros → ε=0.001)\n")
EPS = 1e-3
DP_safe = np.where(DP == 0, EPS, DP)
show_support(DP_safe.prod(axis=0), "Product Rule")

# 4f. Weighted Average
sub("4f. Weighted Average")
print("  Formula: W_j = Σᵢ wᵢ·p_{i,j}\n")
raw_wa = {name: (1 - ERROR[name]) / ERROR[name] for name in MODELS}
total_wa = sum(raw_wa.values())
WA = np.array([raw_wa[n] / total_wa for n in names_list])
show_support(WA @ DP, "Weighted Average")

# 4g. Probabilistic Product
sub("4g. Probabilistic Product")
print("  Formula: W_j = Πᵢ p_{i,j} / P(vⱼ)^(L-1)\n")
prior = np.array([(y_train == j).sum() / len(y_train) for j in range(C)])
prior = np.where(prior == 0, EPS, prior)
print(f"  Class priors P(vⱼ): {[f'{prior[j]:.3f}' for j in range(C)]}")
show_support(DP_safe.prod(axis=0) / (prior ** (L - 1)), "Probabilistic Product")

# 4h. Decision Templates
sub("4h. Decision Templates")
print("  Training: DT_j = mean of DP(x) over all training samples of class j\n")
DT = {}
for j in range(C):
    mask = (y_train == j)
    if mask.sum() == 0: continue
    dps = np.array([clf.predict_proba(X_train_scaled[mask]) for clf in MODELS.values()])
    DT[j] = dps.mean(axis=0)  # shape (L, C)

print("\n  Euclidean similarity: s = 1 − (1/L·C) Σ(DP − DT_j)²")
W_dt = np.zeros(C)
for j in range(C):
    if j not in DT:
        W_dt[j] = 0.0
        continue
    diff = DP - DT[j]
    sq = (diff ** 2).sum()
    W_dt[j] = 1 - sq / (L * C)
show_support(W_dt, "Decision Templates")

# 4i. Dempster-Shafer
sub("4i. Dempster-Shafer (Theory of Evidence)")
print("  Step 1: φ_{j,m} = proximity of classifier m to template j\n")
phi = np.zeros((C, L))
for m in range(L):
    dists = np.array([np.linalg.norm(DT[j][m] - DP[m]) ** 2 for j in range(C)])
    inv   = 1.0 / (1.0 + dists)
    phi[:, m] = inv / inv.sum()

print("  Step 2: Bel_j(m) = belief degree\n")
bel = np.zeros((C, L))
EPS_DS = 1e-9
for j in range(C):
    for m in range(L):
        phi_jm      = phi[j, m]
        prod_others = np.prod([1 - phi[k, m] for k in range(C) if k != j])
        numerator   = phi_jm * prod_others
        denominator = 1 - phi_jm * (1 - prod_others) + EPS_DS
        bel[j, m]   = numerator / denominator

print("  Step 3: µ_j = K · Πₘ Bel_j(m)\n")
ds_raw = np.prod(bel, axis=1)
K      = 1.0 / ds_raw.sum() if ds_raw.sum() > 0 else 1.0
show_support(K * ds_raw, "Dempster-Shafer")

# ═══════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════
header("FINAL SUMMARY")
print(f"\n  True class : {TRUE_LABEL}")
print(f"  {'Method':<35} {'Decision'}")
print(f"  {'-'*50}")

results = [
    ("Abstract — Majority Vote",        mv_winner),
    ("Abstract — Weighted Maj. Vote",   wmv_winner),
    ("Rank    — Borda Count",           bc_winner),
    ("Rank    — Highest Rank",          hr_winner),
    ("Rank    — Intersection",          inter_winner),
    ("Rank    — Union",                 ', '.join(sorted(union_result))),
    ("Measure — Max Rule",              CLASSES[int(np.argmax(DP.max(axis=0)))]),
    ("Measure — Min Rule",              CLASSES[int(np.argmax(DP.min(axis=0)))]),
    ("Measure — Median Rule",           CLASSES[int(np.argmax(np.median(DP, axis=0)))]),
    ("Measure — Sum Rule",              CLASSES[int(np.argmax(DP.sum(axis=0)))]),
    ("Measure — Product Rule",          CLASSES[int(np.argmax(DP_safe.prod(axis=0)))]),
    ("Measure — Weighted Average",      CLASSES[int(np.argmax(WA @ DP))]),
    ("Measure — Probabilistic Product", CLASSES[int(np.argmax(DP_safe.prod(axis=0) / (prior ** (L - 1))))]),
    ("Measure — Decision Templates",    CLASSES[int(np.argmax(W_dt))]),
    ("Measure — Dempster-Shafer",       CLASSES[int(np.argmax(K * ds_raw))]),
]

for method, decision in results:
    correct = "✓" if TRUE_LABEL == decision else "✗"
    print(f"  {method:<35} {{{decision}}}  {correct}")

print(f"\n{SEP}")